# Comprender el problema y los datos de recomendación de cultivos

Antes de calcular estadísticas o construir gráficos, necesitamos entender qué problema representa el conjunto de datos, qué significa cada columna y qué decisión se espera apoyar.

Este notebook es introductorio: describe el contexto y el significado de los datos. La exploración estadística y las métricas se desarrollarán en otros notebooks.

## ¿Qué problema se quiere resolver?

El conjunto de datos está organizado para asociar las condiciones de un terreno o cultivo con una categoría de cultivo (`label`). La pregunta que representa es:

> Dadas las características del suelo y algunas condiciones ambientales, ¿qué tipo de cultivo aparece asociado a esas condiciones en los datos?

En términos de Machine Learning, esto plantea un problema de **clasificación supervisada**: se proporcionan variables de entrada y una etiqueta categórica que sirve como respuesta objetivo. La finalidad es aprender esa asociación para apoyar una recomendación; el resultado de un modelo no reemplaza la evaluación de un agrónomo ni garantiza el rendimiento de una cosecha.

### ¿Por qué puede ser importante?

El cultivo adecuado depende de que sus requerimientos sean compatibles con el suelo y el ambiente. Usar información como nutrientes, temperatura, humedad, acidez y lluvia puede ayudar a orientar una decisión y evitar recomendaciones basadas en una sola condición. Por eso importa comprender qué representan las columnas y qué información no está incluida antes de entrenar un modelo.

## ¿Qué contiene la base de datos?

El archivo `data/raw/Crop_recommendation.csv` es una tabla. Cada fila es un ejemplo compuesto por:

- **Variables de entrada:** `N`, `P`, `K`, `temperature`, `humidity`, `ph` y `rainfall`. Describen propiedades del suelo y condiciones ambientales asociadas al ejemplo.
- **Variable objetivo:** `label`. Indica la categoría de cultivo asociada a esas condiciones.

La tabla no incluye identificador de parcela, ubicación, fecha, temporada ni método de medición. Por eso, no debemos asumir que una fila identifica una finca concreta o una observación con contexto geográfico y temporal conocido.

## Diccionario de datos

| Columna | Significado | Papel en el problema | Unidad / interpretación |
|---|---|---|---|
| `N` | Nivel de nitrógeno disponible en el suelo | Entrada | El CSV no especifica unidad, profundidad ni método de medición. |
| `P` | Nivel de fósforo disponible en el suelo | Entrada | El CSV no especifica unidad, profundidad ni método de medición. |
| `K` | Nivel de potasio disponible en el suelo | Entrada | El CSV no especifica unidad, profundidad ni método de medición. |
| `temperature` | Temperatura asociada al ejemplo | Entrada | Habitualmente se interpreta en °C en este conjunto, pero la unidad no está escrita en el CSV. |
| `humidity` | Humedad asociada al ejemplo | Entrada | Habitualmente se interpreta como porcentaje; el CSV no declara explícitamente la unidad. |
| `ph` | Medida de acidez o alcalinidad del suelo | Entrada | Escala de pH, sin unidad física. Valores menores de 7 son ácidos, 7 es neutro y mayores de 7 son alcalinos. |
| `rainfall` | Lluvia asociada al ejemplo | Entrada | Habitualmente se interpreta en milímetros; el CSV no especifica periodo de acumulación ni unidad. |
| `label` | Nombre del cultivo asociado a las condiciones | Objetivo | Categoría textual, por ejemplo `rice` o `maize`. |

**Precaución sobre unidades:** el archivo contiene nombres y números, pero no incluye metadatos ni documentación de medición. Las convenciones de temperatura, humedad y lluvia son interpretaciones habituales de este dataset; antes de aplicar el modelo a datos reales, habría que confirmar las unidades, periodo, ubicación y protocolo en la documentación de origen. No se debe inventar una unidad para `N`, `P` o `K`.

## Leer un ejemplo y las categorías posibles

La siguiente celda solo carga el CSV, muestra unos registros para relacionar el diccionario con filas reales y lista los nombres de cultivo que aparecen como etiqueta. No calcula estadísticas ni métricas.

In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
data_path = project_root / "data" / "raw" / "Crop_recommendation.csv"
if not data_path.exists():
    project_root = project_root.parent
    data_path = project_root / "data" / "raw" / "Crop_recommendation.csv"

if not data_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo de datos: {data_path}")

df = pd.read_csv(data_path)

print("Ejemplo de una fila (un conjunto de condiciones y su cultivo asociado):")
display(df.head(3))

crop_categories = sorted(df["label"].dropna().unique())
print("\nCategorías de cultivo presentes en label:")
print(", ".join(crop_categories))

Ejemplo de una fila (un conjunto de condiciones y su cultivo asociado):


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice



Categorías de cultivo presentes en label:
apple, banana, blackgram, chickpea, coconut, coffee, cotton, grapes, jute, kidneybeans, lentil, maize, mango, mothbeans, mungbean, muskmelon, orange, papaya, pigeonpeas, pomegranate, rice, watermelon


## Cómo interpretar una fila

En el ejemplo, una fila contiene los niveles registrados de `N`, `P` y `K`, valores de temperatura, humedad, pH y lluvia, y termina con `label = rice`. Esto significa que, en este conjunto, ese perfil de condiciones está asociado a la categoría `rice`.

La fila **no** demuestra por sí sola que esas condiciones causen el crecimiento del arroz, ni que garanticen una cosecha exitosa. Tampoco permite saber dónde o cuándo se midieron las condiciones, porque esa información no aparece en las columnas.

## Qué información falta y por qué importa

El conjunto se enfoca en siete variables de suelo y ambiente, pero una decisión agrícola real puede depender también de ubicación, estación, tipo y profundidad del suelo, disponibilidad de riego, variedad de semilla, plagas, costos y condiciones de mercado. Esos factores no aparecen en esta tabla.

Además, el CSV no documenta quién generó las etiquetas, cómo se eligieron los cultivos asociados ni el protocolo de medición. Antes de usar estos datos para decisiones reales, habría que consultar la documentación de origen y validar que las unidades y el contexto sean compatibles con el lugar de aplicación.

### Límite de este notebook

Aquí entendemos el objetivo, la estructura y el significado de las columnas. No evaluamos calidad, distribuciones, correlaciones ni rendimiento de modelos. Ese análisis se realizará después, una vez que sepamos qué representa cada variable y qué supuestos debemos comprobar.